In [11]:
import yaml
from app.datasets.loader import load_multiple_test_cases, load_test_cases
from app.datasets.validator import validate_dataset_schema
from app.client.rag_client import RAGClient
from app.tests.run_tests import run_tests
from app.tests.nodes.reformulate import send_reformulate_requests, save_reformualte_responses, reformulate_tests

In [12]:
file_list = [
  './app/data/raw/rapido.xlsx',
]

test_config = {
  'GENERAL_TESTS': True,
  'TIMINGS': {'test': True, 'report': False},
  'TOKENS': {'test': True, 'report': False},
  'FOUNDRYS': {'test': True, 'report': False},
  'TRIAGE': {'test': True, 'report': False},
  'ROUTER': {'test': True, 'report': False},
  'GROUNDING': {'test': True, 'report': False},
  'SAVE_RESULTS': False,
  'PATH': './app/data/processed/reports/report_RAPIDO',
  
  'REFORMULATE': {'test': True, 'report': False}
}   

if file_list: 
  df = load_multiple_test_cases(file_list)
  df = validate_dataset_schema(df)

with open('./app/config/config.yaml', 'r') as file:
  config_data = yaml.load(file, Loader= yaml.FullLoader) 
  
client = RAGClient(config_data)
test_timestamps = {}

In [14]:
# TEST GENERALES
if test_config.get('GENERAL_TESTS', False):
  responses = client.query_batch(df['user_input'],df['reference'])
  save_responses_in_json, response_file_path = client.save_api_responses(responses)
  test_timestamps['general_tests'] = str(response_file_path).replace('\\', '/')

Processing queries:  70%|███████   | 7/10 [01:18<00:38, 12.68s/it]

Max retries exceeded
Network Error: 'NoneType' object has no attribute 'get'


Processing queries:  80%|████████  | 8/10 [01:40<00:31, 15.58s/it]

Max retries exceeded
Network Error: 'NoneType' object has no attribute 'get'


Processing queries: 100%|██████████| 10/10 [01:52<00:00, 11.25s/it]


In [59]:
import json

response_file_path = './app/data/processed/outcomes/outcome_20260412-204045.json'
with open(response_file_path, 'r', encoding='UTF-8') as f:
  responses = json.load(f)
test_timestamps['general_tests'] = 'outcome_20260412-204045.json'

In [53]:
if test_config.get('GENERAL_TESTS'):
  results, reports = run_tests(
    config = test_config, 
    data = responses, 
    df = df, 
    timestamp = test_timestamps
)

In [64]:
print(results)

{'timestamp': '20260412-204045', 'nodes': {'triage': {'positives': 8, 'total': 8, 'result': 100.0}, 'router': {'positives': 8, 'total': 8, 'result': 100.0}, 'grounding': {'positives': 8, 'total': 8, 'result': 100.0}}, 'timings': {'reformulate': {'prom': 0.779, 'p90': 1.073, 'p95': 1.077, 'quantity': 8}, 'triage': {'prom': 0.941, 'p90': 1.1, 'p95': 1.119, 'quantity': 8}, 'router': {'prom': 0.778, 'p90': 1.61, 'p95': 2.947, 'quantity': 8}, 'ag_call': {'prom': 3.008, 'p90': 4.537, 'p95': 5.678, 'quantity': 8}, 'personality': {'prom': 1.672, 'p90': 2.505, 'p95': 2.748, 'quantity': 8}, 'grounding': {'prom': 1.516, 'p90': 2.107, 'p95': 2.352, 'quantity': 8}, 'retriever': {'prom': 1.075, 'p90': 2.352, 'p95': 3.542, 'quantity': 8}, 'ret_embeddings': {'prom': 0.791, 'p90': 2.066, 'p95': 3.226, 'quantity': 8}, 'rag_answer': {'prom': 1.617, 'p90': 2.176, 'p95': 2.194, 'quantity': 8}, 'response_time': {'prom': 7.056, 'p90': 9.435, 'p95': 9.655, 'quantity': 8}}, 'tokens': {'in_ref': {'prom': 929.25